# ACME: raw ROS 2 bags

Each robot trajectory is also released as the original ROS 2 (Humble, sqlite3) bag,
zipped: `ROS2_Bags/<institution>/<bag>.zip`. The bags hold every recorded topic plus
the anonymization outputs (`/tracks_*` pedestrian boxes, `/masks_*` person masks,
`/keypoints_*` body keypoints).

`acme_dataset.rosbag` reads them with the pure-Python `rosbags` library, so **no ROS
installation is needed**.

In [ ]:
from pathlib import Path
import os

# Root of the dataset: the folder that contains `ROS2_Bags/`, `sampled_data/`,
# `BEV/` and `acme_tags.xlsx`. Override with the ACME_ROOT environment variable.
DATA_ROOT = Path(os.environ.get("ACME_ROOT", "../ACME-dataset"))

DOWNLOAD = False
if DOWNLOAD:
    from acme_dataset import download, list_remote
    names = list_remote("raw", institution="miraikan")
    download("raw", "miraikan", names[:1], local_dir=DATA_ROOT)   # one bag, ~100 MB

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from acme_dataset.rosbag import find_bags, metadata_summary

bags = find_bags(DATA_ROOT / "ROS2_Bags")
print(bags)
BAG = bags[0]

# topic list straight from metadata.yaml (no extraction needed)
duration, topics = metadata_summary(BAG)
print(f"{duration:.1f} s")
topics

## Open the bag

`Bag` extracts the zip next to itself on first use (pass `extract_dir=` to change that).

In [ ]:
from acme_dataset.rosbag import Bag, image_topic_for

bag = Bag(BAG).open()   # or: with Bag(BAG) as bag: ...
topic_names = bag.topics["topic"].tolist()
odom_topic = next(t for t in ("/odom", "/utlidar/robot_odom", "/odom_icp", "/cabot/odometry/filtered",
                              "/go1_controller/odom", "/zed2/zed_node/odom") if t in topic_names)
tracks_topic = next(t for t in topic_names if t.startswith("/tracks_"))
image_topic = image_topic_for(tracks_topic, topic_names)
pcl_topic = next((t for t in topic_names if t in ("/velodyne_points", "/rslidar_points", "/rslidar_helios_points")), None)
odom_topic, image_topic, tracks_topic, pcl_topic

## Odometry

In [ ]:
from acme_dataset.viz import plot_odometry

odom = bag.odometry(odom_topic)
display(odom.head())
plot_odometry(odom)
plt.tight_layout()

## Images and pedestrian boxes

Depending on the anonymization pipeline version, `bbox.center` in a bag is either the
true box center or `(top_left + size) / 2` (legacy). `detect_bbox_encoding` finds out
which one a bag uses, and `decode_detections` returns top-left `[x, y, w, h]` boxes either way.

In [ ]:
import pandas as pd
from acme_dataset.rosbag import decode_compressed_image, decode_detections, detect_bbox_encoding
from acme_dataset.viz import draw_detections

encoding = detect_bbox_encoding(bag, tracks_topic)
print("bbox encoding:", encoding)

t_img, img_msg = bag.read_topic(image_topic, limit=60)[-1]
tracks = bag.read_topic(tracks_topic)
t_trk, trk_msg = min(tracks, key=lambda m: abs(m[0] - t_img))   # closest tracks message
boxes, ids = decode_detections(trk_msg, encoding)
det = pd.DataFrame(boxes, columns=["x", "y", "width", "height"]).assign(track_id=ids)
plt.figure(figsize=(10, 6))
plt.imshow(draw_detections(decode_compressed_image(img_msg), det))
plt.title(f"{len(det)} pedestrians, |dt| = {abs(t_trk - t_img) / 1e6:.0f} ms")
plt.axis("off");

## Person masks and keypoints

In [ ]:
from acme_dataset.rosbag import decode_binary_mask, decode_keypoints

mask_topic = tracks_topic.replace("/tracks_", "/masks_")
kp_topic = tracks_topic.replace("/tracks_", "/keypoints_")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
if mask_topic in topic_names:
    axes[0].imshow(decode_binary_mask(bag.read_topic(mask_topic, limit=1)[0][1]), cmap="gray")
    axes[0].set_title(mask_topic)
if kp_topic in topic_names:
    people = decode_keypoints(bag.read_topic(kp_topic, limit=1)[0][1])
    for person in people:
        visible = person[:, 2] > 0.5
        axes[1].scatter(person[visible, 0], person[visible, 1], s=8)
    axes[1].invert_yaxis()
    axes[1].set_title(f"{kp_topic}: {len(people)} people (COCO-17 keypoints)")
for ax in axes:
    ax.set_aspect("equal")

## LiDAR, calibration

In [ ]:
from acme_dataset.rosbag import decode_pointcloud2
from acme_dataset.viz import plot_pointcloud_bev

if pcl_topic:
    points = decode_pointcloud2(bag.read_topic(pcl_topic, limit=1)[0][1])
    print(points.shape)
    plot_pointcloud_bev(points, x_range=(-10, 10), y_range=(-10, 10))

display(bag.static_transforms().head(10))   # sensor extrinsics from /tf_static
display(bag.camera_info())                   # intrinsics of every CameraInfo topic

## Re-sample a bag into the human-readable format

`convert_bag` (or `acme-dataset convert`) produces exactly the folder layout of
`sampled_data/<institution>/<trajectory>`, at any rate. The topics per institution are listed in
`acme_dataset/configs/topics.yaml`.

In [ ]:
import tempfile
from acme_dataset import Trajectory
from acme_dataset.convert import convert_bag

out = convert_bag(BAG, tempfile.mkdtemp(), rate=4.0)
Trajectory(out).summary()

In [ ]:
bag.close()